# Setting up Python
## Installing libraries
Use `%pip install <libraryname>` to install libraries. You can add your own library names to the cell and re-run to install extra libraries.


In [ ]:
%pip install minio

Restart the kernel to allow loading of the installed libraries

In [ ]:
import os
os._exit(00)

## Notebook functions, libraries and environmental variables

In [ ]:
# %% Libraries
import pathlib
import getpass
from minio import Minio
import configparser

# %% Functions
def get_app_config_dir(appname):
    app_rel_path = f".config/{appname}/{appname}.conf"
    app_system_path = pathlib.Path.home().joinpath(app_rel_path)
    return app_system_path

# %% Env Var
appname = "oceananalysis"
server_config_name = "default"

# Setting Abyssio configuration

In [ ]:
# %% Main
app_system_path = get_app_config_dir(appname = appname) 
app_system_path.parent.mkdir(parents=True, exist_ok=True)
if app_system_path.exists():
    answer_options = ["y","n"]
    answer = ""
    while True:
        answer = input(f"{app_system_path} already exists, do you want to overwrite? [{answer_options}]")
        if answer == answer_options[0]:
            write_file = True
            break
        if answer == answer_options[1]:
            write_file = False
            break
        else:
            print(f"Answer was not one of the available options, please us either of the answer options: {answer_options}")
else:
    write_file = True

if write_file:
    profile = 'default'
    access_key = input("Enter access key")
    secret_key = getpass.getpass('Enter secret key')
    config_data = f"""\
    [{profile}]
    access_key = {access_key}
    secret_key = {secret_key}
    endpoint = 145.38.204.177:9000
    secure = False
    """
    print(f"Writing configuration file to {app_system_path}")
    app_system_path.write_text(config_data)
    print("Done!")

# Testing connection 

In [ ]:
# %% Main
# Read configuration
app_system_path = get_app_config_dir(appname = appname) 
config_parser = configparser.ConfigParser()
config_parser.read(app_system_path)
# Set up minio Client
client = Minio(
    endpoint=config_parser.get(server_config_name,"endpoint"),
    access_key=config_parser.get(server_config_name,"access_key"),
    secret_key=config_parser.get(server_config_name,"secret_key"),
    secure = config_parser.getboolean(server_config_name,"secure")
)
# Test if we can connect
buckets = client.list_buckets()
for bucket in buckets:
    print(f"{bucket.name}")

# Examples using Abyssio

## Setting your connection

In [ ]:
# %% Main
# Read configuration
app_system_path = get_app_config_dir(appname = appname) 
config_parser = configparser.ConfigParser()
config_parser.read(app_system_path)
# Set up minio Client
client = Minio(
    endpoint=config_parser.get(server_config_name,"endpoint"),
    access_key=config_parser.get(server_config_name,"access_key"),
    secret_key=config_parser.get(server_config_name,"secret_key"),
    secure = config_parser.getboolean(server_config_name,"secure")
)

## Downloading data

### Downloading a single file

In [ ]:
# Downloading a single file
# %% Libraries
import pathlib
# %% Env var
# Server location information
bucket_name = "input"
prefix = "netcdf"
file_name = "CO2f_fesom_historical_19900101.nc"
object_name = f"{prefix}/{file_name}"
# Local location information
local_download_directory = "downloads"
local_download_path = pathlib.Path.home().joinpath(local_download_directory)
local_file_path = f"{local_download_path.as_posix()}/{file_name}"
if not local_download_path.exists():
    local_download_path.mkdir(parents = True, exist_ok = False)
# %% Main
# Downloading the file
client.fget_object(bucket_name = bucket_name,
                  object_name = object_name,
                  file_path = local_file_path)

### Downloading a full prefix (directory)

In [ ]:
# Downloading a full prefix recursively
# %% Libraries
import pathlib
import time
stime = time.time()

# %% Env var
# Server location information
bucket_name = "input"
prefix = "netcdf/" # Trailing slash required to 'look into prefix'

# Local location information
local_download_directory = "downloads"
local_download_path = pathlib.Path.home().joinpath(local_download_directory)
if not local_download_path.exists():
    local_download_path.mkdir(parents = True, exist_ok = False)

# %% Main
# List objects in prefix
objects = client.list_objects(bucket_name = bucket_name,
                   prefix = prefix,
                   recursive = False) # Recursive set to false, only looking in this prefix

# Lets see how well we did, lets track some stats.
nobjs = 0
bytes_downloaded = 0

# Download the objects
for obj in objects: # This is a generator, it 'consumes'. If you want to look back into objects. turn it into a list first.
    local_file_path = f"{local_download_path.as_posix()}/{obj.object_name}"
    client.fget_object(bucket_name = obj.bucket_name,
                      object_name = obj.object_name,
                      file_path = local_file_path)
    # Update our stats
    nobjs += 1
    bytes_downloaded += obj.size

# Parse the stats such that we can print a nice message
mbytes_downloaded = bytes_downloaded/(1<<20)
elapsed = time.time() - stime
mbytes_avg_xfer = mbytes_downloaded/elapsed

# Printable results
print(f"Downloaded {nobjs} files, totalling {mbytes_downloaded:.2f} MB in {elapsed:.2f}s with an average speed of {mbytes_avg_xfer:.2f} MB/s")

## Uploading data

### Uploading a single file

In [ ]:
# Create a test file to test uploading data.
testfile_path = pathlib.Path().home().joinpath('testfile.txt')
#testfile_path.write_text('Hello World.')
bucket_name = 'personal'
prefix = "analysis/output"
access_key = config_parser.get(server_config_name,"access_key")
object_name = f"{access_key}/{prefix}/{testfile_path.name}"
client.fput_object(bucket_name = bucket_name,
                  object_name = object_name,
                  file_path = testfile_path)

### Uploading a directory

In [ ]:
# Create a test directory with some files
test_directory = pathlib.Path().home().joinpath('test_dir')
if not test_directory.exists():
    test_directory.mkdir(parents=True,exist_ok = False)
for i in range(0,10):
    test_directory.joinpath(f"file_{i}.txt").write_text(f"This is file {i}")


# Upload the data
bucket_name = 'personal'
prefix = f"analysis/{test_directory.name}"
access_key = config_parser.get(server_config_name,"access_key")
file_list = test_directory.glob("*.txt")
for file_path in file_list:
    object_name = f"{access_key}/{prefix}/{file_path.name}"
    client.fput_object(bucket_name = bucket_name,
                  object_name = object_name,
                  file_path = testfile_path)